# 🛰️ Hands‑On: SMAP & Sentinel‑1 — Runnable Colab

**Afternoon workshop · Part 2.** Companion to the *"From Coarse Pixels to Field‑Scale Detail"* slides.

Today you will:
1. Load real **Sentinel‑1** radar and build a **dual‑polarization (VV/VH) ratio**
2. Compute a **Radar Vegetation Index (RVI)**
3. Map a real flood — the **March 2019 Missouri River floods**
4. View **SMAP soil moisture (9 km)** over the **U.S. Corn Belt**

**One region all afternoon:** the U.S. Corn Belt — central Iowa for the radar exercises, the Nebraska/Iowa Missouri River for the flood, and the whole belt for soil moisture.

> **Two logins today:** Google **Earth Engine** (Exercises 1–3) and a free **NASA Earthdata** account (Exercise 4). Run the cells top to bottom; edit the AOIs in the setup cell to study your own area.


> ### ✏️ STUDENT copy — fill in the blanks
> Wherever you see `# TODO` and `____`, write the missing line yourself, then run the cell. Hints are in the markdown above each exercise. Stuck? Check the **SOLUTION** copy. Learn by typing it — don't just copy‑paste!


## 0 · Setup — connect to Earth Engine
Run these two cells once. If `geemap` is already installed, the first cell is instant.


In [ ]:
# Installs the two libraries Colab doesn't ship with
!pip install -q geemap earthaccess


In [ ]:
import ee, geemap

PROJECT_ID = "ee-arunavnanda7"          # <-- demo project; REPLACE with YOUR OWN project id

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()                 # opens a sign-in link; paste the token back
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine ready:", ee.String("connected").getInfo())


In [ ]:
# --- Areas of interest (edit freely to study your own region) ---
# Exercises 1-2: an agricultural patch in central Iowa (corn / soybean mosaic)
AG_AOI    = ee.Geometry.Rectangle([-93.9, 41.9, -93.2, 42.5])

# Exercise 3: Missouri River floodplain on the Nebraska / Iowa border
FLOOD_AOI = ee.Geometry.Rectangle([-96.3, 40.4, -95.4, 41.4])

# Exercise 4: U.S. Corn Belt bounding box  (lon W, lat S, lon E, lat N)
CORN_BELT = (-99, 37, -82, 46)

Map = geemap.Map(center=[42.1, -93.6], zoom=9)
Map


## 1 · Sentinel‑1 backscatter + dual‑pol ratio
Sentinel‑1 sends and receives microwaves in two polarizations: **VV** (sensitive to roughness and open water) and **VH** (cross‑pol, sensitive to vegetation *volume scattering*). Earth Engine delivers them already calibrated, in **decibels (dB)**.

Because the data are in dB, the ratio \(\sigma^0_{VV}/\sigma^0_{VH}\) becomes a simple **subtraction**.


In [ ]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(AG_AOI)
      .filterDate('2025-06-01', '2025-06-30')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')))

print('scenes found:', s1.size().getInfo())
img = s1.median().clip(AG_AOI)                 # one clean monthly composite (dB)

# TODO: the VV/VH ratio in dB is a SUBTRACTION (VV minus VH). Make an image 'ratio':
ratio = ____   # hint: img.select('VV').subtract( ... ).rename('VV_VH')

Map = geemap.Map(center=[42.2, -93.55], zoom=10)
Map.addLayer(img.select('VV'), {'min': -25, 'max': 0},  'VV (dB)')
Map.addLayer(img.select('VH'), {'min': -30, 'max': -5}, 'VH (dB)')
Map.addLayer(ratio, {'min': 0, 'max': 15,
             'palette': ['#2c7bb6', '#ffffbf', '#d7191c']}, 'VV/VH ratio (dB)')
Map


## 2 · Radar Vegetation Index (RVI)
RVI uses the same two channels to estimate vegetation density from volume scattering:

$$\mathrm{RVI} = \frac{4\,\sigma^0_{VH}}{\sigma^0_{VV} + \sigma^0_{VH}}$$

**Crucial:** RVI is defined on *linear power*, but Earth Engine gives dB. Convert first with \(\sigma^0 = 10^{(\sigma^0_{dB}/10)}\). Low RVI → bare/smooth; high RVI → dense canopy. *(RVI comes from Sentinel‑1, not from SMAP.)*


In [ ]:
lin = ee.Image(10).pow(img.divide(10))          # dB -> linear power
vv, vh = lin.select('VV'), lin.select('VH')

# TODO: apply the dual-pol RVI formula  4*VH / (VV + VH)  on linear power:
rvi = ____   # hint: vh.multiply(4).divide( vv.add(vh) ).rename('RVI')

Map = geemap.Map(center=[42.2, -93.55], zoom=10)
Map.addLayer(rvi, {'min': 0, 'max': 1,
             'palette': ['#8c510a','#d8b365','#f6e8c3','#c7eae5','#5ab4ac','#01665e']},
             'RVI')
print('RVI 5/50/95th percentiles:',
      rvi.reduceRegion(ee.Reducer.percentile([5, 50, 95]), AG_AOI, scale=30).getInfo())
Map


## 3 · Map a real flood — March 2019 Missouri River
In March 2019 a "bomb cyclone" caused catastrophic flooding along the Missouri River (Nebraska/Iowa) — right inside the Corn Belt. Radar is perfect for this: it sees through clouds, and **calm open water looks very dark** (it mirrors energy away).

**Method:** compare a dry *before* composite with an *after* composite and flag pixels that became much darker:

$$\Delta = \sigma^0_{\text{after}} - \sigma^0_{\text{before}} \;\;(\text{dB})$$


In [ ]:
def s1_vh(aoi, d0, d1):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi).filterDate(d0, d1)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select('VH').median().clip(aoi))

pre  = s1_vh(FLOOD_AOI, '2019-03-01', '2019-03-12')   # before the bomb cyclone
post = s1_vh(FLOOD_AOI, '2019-03-16', '2019-03-27')   # peak flooding

# TODO: change = after - before
diff  = ____   # hint: post.subtract(pre)
# TODO: flag flood = it is DARK now (post < -18 dB) AND it DROPPED a lot (diff < -3 dB)
flood = ____   # hint: post.lt(-18).And(diff.lt(-3))

Map = geemap.Map(center=[40.95, -95.85], zoom=10)
Map.addLayer(pre,  {'min': -30, 'max': -5}, 'Before VH (Mar 1-12)')
Map.addLayer(post, {'min': -30, 'max': -5}, 'After VH (Mar 16-27)')
Map.addLayer(flood.selfMask(), {'palette': ['#1f6feb']}, 'Flooded (detected)')
Map


> **Try it:** toggle *Before VH* and *After VH* on/off to watch the river burst its banks, then reveal the blue *Flooded* layer. The thresholds (−18 dB, −3 dB) are starting points — smooth dry soil and permanent water can fool them, so this is a screening tool, not a legal flood map.


## 4 · SMAP soil moisture (9 km) over the Corn Belt
Now we switch from radar texture to calibrated **soil moisture**. SMAP products are **not** in Earth Engine — we pull them from NASA NSIDC with `earthaccess` (the same workflow behind this morning's animations).

First, the official SMAP volumetric‑soil‑moisture colour palette, then one real 9 km scene.

> **Earthdata login:** use the **same** NASA Earthdata username & password you already have — there is no separate signup and no portal selections to make. `earthaccess.login(strategy="interactive", persist=True)` prompts you for those credentials in the cell, and `persist=True` saves them for the session so you only type them once.


In [ ]:
# Official SMAP VSM colour palette (256 colours, dry -> wet)
from matplotlib.colors import ListedColormap
SMAP_RGB = [
    (255,161,0), (255,162,0), (255,165,0), (255,167,0), (255,170,0), (255,172,0), (255,174,0), (255,177,0),
    (255,179,0), (255,181,0), (255,184,0), (255,186,0), (255,189,0), (255,191,0), (255,193,0), (255,196,0),
    (255,198,0), (255,200,0), (255,203,0), (255,205,0), (255,208,0), (255,210,0), (255,212,0), (255,215,0),
    (255,217,0), (255,219,0), (255,222,0), (255,224,0), (255,227,0), (255,229,0), (255,231,0), (255,234,0),
    (255,236,0), (255,238,0), (255,241,0), (255,243,0), (255,246,0), (255,248,0), (255,249,0), (255,250,0),
    (255,251,0), (255,253,0), (255,255,0), (255,254,0), (249,252,0), (243,249,0), (237,246,0), (231,243,0),
    (225,240,0), (219,237,0), (213,234,0), (208,231,0), (202,228,0), (196,225,0), (190,222,0), (184,219,0),
    (178,216,0), (172,213,0), (166,210,0), (160,207,0), (154,204,0), (148,201,0), (142,198,0), (136,195,0),
    (130,193,0), (125,190,0), (119,187,0), (113,184,0), (107,181,0), (101,178,0), (95,175,0), (89,172,0),
    (83,169,0), (77,166,0), (71,163,0), (65,160,0), (59,157,0), (53,154,0), (47,151,0), (42,148,0),
    (36,145,0), (30,142,0), (24,139,0), (18,136,0), (12,133,0), (6,130,0), (0,130,0), (0,130,6),
    (0,133,12), (0,136,19), (0,139,25), (0,143,31), (0,146,37), (0,149,44), (0,152,50), (0,155,56),
    (0,158,62), (0,161,68), (0,164,75), (0,168,81), (0,171,87), (0,174,93), (0,177,100), (0,180,106),
    (0,183,112), (0,186,118), (0,189,124), (0,193,131), (0,196,137), (0,199,143), (0,202,149), (0,205,155),
    (0,208,162), (0,211,168), (0,214,174), (0,218,180), (0,221,187), (0,224,193), (0,227,199), (0,230,205),
    (0,233,211), (0,236,218), (0,239,224), (0,243,230), (0,246,236), (0,249,243), (0,252,249), (0,254,255),
    (0,255,255), (0,253,255), (0,249,255), (0,243,255), (0,237,255), (0,231,255), (0,225,255), (0,219,255),
    (0,213,255), (0,208,255), (0,202,255), (0,196,255), (0,190,255), (0,184,255), (0,178,255), (0,172,255),
    (0,166,255), (0,160,255), (0,154,255), (0,148,255), (0,142,255), (0,136,255), (0,130,255), (0,125,255),
    (0,119,255), (0,113,255), (0,107,255), (0,101,255), (0,95,255), (0,89,255), (0,83,255), (0,77,255),
    (0,71,255), (0,65,255), (0,59,255), (0,53,255), (0,47,255), (0,42,255), (0,36,255), (0,30,255),
    (0,24,255), (0,18,255), (0,12,255), (0,6,255), (0,0,255), (0,0,251), (0,0,248), (0,0,244),
    (0,0,241), (0,0,237), (0,0,233), (0,0,230), (0,0,226), (0,0,223), (0,0,219), (0,0,215),
    (0,0,212), (0,0,208), (0,0,205), (0,0,201), (0,0,197), (0,0,194), (0,0,190), (0,0,187),
    (0,0,183), (0,0,179), (0,0,176), (0,0,172), (0,0,168), (0,0,165), (0,0,161), (0,0,158),
    (0,0,154), (0,0,150), (0,0,147), (0,0,143), (0,0,140), (0,0,136), (0,0,132), (0,0,129),
    (0,0,125), (0,0,122), (0,0,118), (0,0,114), (0,0,111), (0,0,107), (0,0,104), (0,0,100),
    (2,2,100), (4,4,100), (6,6,100), (8,8,100), (10,10,100), (12,12,100), (14,14,100), (16,16,100),
    (18,18,100), (20,20,100), (22,22,100), (24,24,100), (26,26,100), (28,28,100), (30,30,100), (32,32,100),
    (34,34,90), (36,36,90), (38,38,90), (40,40,90), (42,42,90), (44,44,90), (46,46,90), (48,48,90),
    (50,50,90), (52,52,90), (54,54,90), (56,56,90), (58,58,90), (60,60,80), (62,62,80), (64,64,80),
    (66,66,80), (68,68,80), (70,70,80), (72,72,80), (74,74,80), (76,76,80), (78,78,80), (80,80,80),
]
smap_cmap = ListedColormap([(r/255, g/255, b/255) for r, g, b in SMAP_RGB], 'smap_vsm')
print('palette ready:', len(SMAP_RGB), 'colours')


In [ ]:
# SMAP L3 Enhanced 9 km (SPL3SMP_E) via earthaccess  (needs a free NASA Earthdata login)
import earthaccess, h5py, numpy as np, matplotlib.pyplot as plt

earthaccess.login(strategy="interactive", persist=True)   # SAME Earthdata username/password

W, S, E, N = CORN_BELT
res = earthaccess.search_data(short_name='SPL3SMP_E',
        temporal=('2025-06-15', '2025-06-16'),
        bounding_box=(W, S, E, N))
files = earthaccess.download(res[:1], 'smap_9km')
print('downloaded:', files)

g = 'Soil_Moisture_Retrieval_Data_AM/'
with h5py.File(files[0], 'r') as f:
    sm  = f[g + 'soil_moisture'][:]
    lat = f[g + 'latitude'][:]
    lon = f[g + 'longitude'][:]

sm  = np.where(sm  <= -9000, np.nan, sm)
lat = np.where(lat <= -9000, np.nan, lat)
lon = np.where(lon <= -9000, np.nan, lon)

# TODO: keep only valid points INSIDE the Corn Belt box (use W,S,E,N and np.isfinite)
m = ____
plt.figure(figsize=(11, 6))
sc = plt.scatter(lon[m], lat[m], c=sm[m], s=9, cmap=smap_cmap, vmin=0, vmax=0.6)
plt.title('SMAP L3 Enhanced 9 km — surface soil moisture\nU.S. Corn Belt, 15 Jun 2025')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
cb = plt.colorbar(sc, orientation='horizontal', pad=0.12, fraction=0.05)
cb.set_label('Volumetric soil moisture (m3/m3)    dry -> wet')
plt.tight_layout(); plt.show()


### The 3 km and 1 km products
The finer SMAP/Sentinel‑1 products use the same `earthaccess` path with **`short_name='SPL2SMAP_S'`** (read `Soil_Moisture_Retrieval_Data_3km/soil_moisture_3km` or the 1 km group). They are heavier, so for class we view the **ready‑made monthly animations** instead — see the concepts page.

**What to notice:** 9 km is smooth and complete; 1 km is sharp but patchier (it needs a Sentinel‑1 overpass). Coverage therefore *increases* 1 km → 3 km → 9 km.


## ✅ Wrap‑up
You loaded radar, made a vegetation index, mapped a real flood, and viewed satellite soil moisture — all over one landscape.

**Take‑home:** the full research notebook (download → process → 1 km disaggregation) is at
👉 https://github.com/Arunavmsu07/SAR_Workshop

**Mini‑projects:** re‑run Exercises 1–2 over your hometown · find a recent flood and set the AOI/dates in Exercise 3 · build SMAP monthly means for one belt across a whole year.
